# Birth-Death Skyline (BDSKY) serial workflow


This Workflow Notebook is for running BDSKY serial models in BEAST 2. **The template BEAST 2 xml (template_xml) or ready_to_go_xml provided must be for BDSKY serial!**

<details>
    <summary>Click To See A Decription of Parameters</summary>
        <pre>
            <code>

Running an Instance of this Workflow
-------------------------------------------
overall_save_dir: str
    Path to where you are saving all the runs of this workflow.

specific_run_save_dir: str, optional
    Sub-directory of overall_save_dir you wish to save the outputs from this workflow.
    If None, 'None' or an empty string a timestamp of format 'YYYY-MM-DD_hour-min-sec' is used instead.

max_threads: int, default None
    The maximum number of threads to use when calling gnu parallel in phases 2i and 4. If None and BEAST_pype is running
    in a SLURM job the SLURM environment variable `SLURM_CPUS_PER_TASK` is used. If None and BEAST_pype is NOT running in
    a SLURM job the number of cores available minus 1 is used (`multiprocessing.cpu_count() - 1`).

kernel_name: str, default 'beast_pype'
    Name of Jupyter python kernel to use when running workflow. This is also the name of the conda environment to use in phases 4 &
    phase 2ii (as these Jupyter notebooks use the `bash` kernel).

start_timeout: int, default 60
    Seconds to wait for a notebook to start before raising an error. This is used in all phases. If not provided the default is 60 seconds (1 minute).

General Inputs
----------------
ready_to_go_xml: str, optional
    Path to a BEAST 2 xml that you wish to run unaltered. If provided phases 2i, 2ii and 3 are skipped.

template_xml_path: str
    Path to template BEAST 2 xml.

fasta_path: str
    Path to fasta file containing sequences to be placed into template xml.

metadata_path: str
      Path to csv or tsv containing metadata for sequences in fasta_path.

sample_id_field: str
    Name of field in metadata_db containing sequence IDs.

collection_date_field: str
    Name of field in metadata_db containing collection dates of sequences. Should be formatted YYYY-MM-DD.

seed_to_rule_them_all: int, optional
    If provided this seed is used to seed a random number generator and all other seeds in the workflow are generated from  that random number generator. Thus, ensuring the same seed is used across all steps of the workflow for reproducibility. If not provided individual seeds can be provided for each step of the workflow that requires a seed (see below).

Add pre-made Initial Tree
--------------------

initial_tree_path: str, optional
    Path to initial tree to use in generating a BEAST 2 xml. Should be .nwk file (Newick format).
    If provided phases 2i and 2ii are skipped.
    If a distance tree is used set initial_tree_type to 'Distance'.
    If a temporal tree is used set initial_tree_type to 'Temporal'.


Initial Tree Building & Downsampling
------------------
use_initial_tree:  bool, default True
    If False an initial tree will not be generated skipping Phases 2i and 2ii. As such, in phase 4 BEAST 2 generate its own
    initial tree.

iqtree_seed: int, optional
    Seed to use when building IQtree distance tree in phase 2i.

treetime_seed: int, optional
    Seed to use when building TreeTime temporal tree and downsampling in phase 2ii.

downsampled_iqtree_seed: int, optional
    Seed to use when building IQtree distance tree in phase 2iii.

downsampled_treetime_seed: int, optional
    Seed to use when building TreeTime temporal tree in phase 2iv

initial_tree_type: str (either 'Distance' or 'Temporal') or None, default 'Temporal'
    Initial tree type to use.
    If 'Distance' and initial_tree_path is not provided the IQtree tree from Phase-2i-IQTree.ipynb is used for the
    initial tree and phase 2ii is skipped.
    if 'Temporal' and initial_tree_path is not provided the TreeTime tree from Phase-2ii-TreeTime-and-Downsampling.ipynb
    is used for the initial tree.

root_strain_names: list of strings, optional
    IDs of sequences used to root 'Temporal' initial_tree. These sequences/nodes are removed from fasta file and initial tree file used to generate the BEAST 2 xml.

remove_root: bool, default False
    If True, remove the root after rerooting. This is useful if the root is not a real sample and is only used for rooting purposes.


downsample_to: int, optional
    If provided the fasta file and initial tree file used to generate the BEAST 2 xml is downsampled to this amount.
    If downsampling occurs the following are saved in  '{overall_save_dir}/{specific_run_save_dir}/' and used in generating
    a BEAST 2 xml in phase 4:
        downsampled_time.nwk: A downsampled temporal tree.
        downsampled_sequences.fasta: Fasta file containing downsampled sequences.
        downsampled_metadata.csv: the down sampled metadata.


BDSky Options
------------------
origin_start_addition: float
    Suggested infection period of pathogen. **Should be in years.** This + initial MLE tree height is used as starting value of origin.

origin_upper_addition: float/int
    This + initial MLE tree height is used as upper value of origin prior.

origin_prior: dict {'lower': float, 'upper': float, 'start': float}, optional
    Details of the origin prior assumed to be uniformly distributed.

rt_dims: int, optional
    Number of Rt dimensions (time periods over which Rt is estimated).

rt_partitions: dict of strings {'unit': 'days, weeks or years', 'every': int/float, 'end': date str YYYY-MM-DD}, optional
    Instructions for setting rt_change date, going backwards from the youngest date in the metadata until rt_partitions["end"]  is reached.
    If rt_partitions["end"] is not given the oldest date in metadata is used for this end point value instead.
    rt_partitions["end"] should be a datetime object or string of format 'YYYY-MM-DD'.
    If given rt_dims must equal None.

sampling_prop_dims: int, optional
    Number of sampling proportion dimensions (time periods over which sampling proportion is estimated).

sampling_prop_partitions: dict of strings {'unit': 'days, weeks or years', 'every': int/float, 'end': date str YYYY-MM-DD}, optional
    Instructions for setting sampling_prop_change date, going backwards from the youngest date in metadata until sampling_prop_partitions["end"]  is reached.
    If sampling_prop_partitions["end"] is not given the oldest date in metadata is used for this end point value instead.
    sampling_prop_partitions["end"] should be a datetime object or string of format 'YYYY-MM-DD'.
    If given sampling_prop_dims must equal None.

zero_sampling_before_first_sample: bool, default False
    If true fix the sampling proportion to 0 for the period before the first sample.

MCMC Tree/Logfile names Chain-lengths & Save Steps
------------------
log_file_basename: str, optional
    If provided .tree, .log and .state files from running BEAST 2 will have this name prefixed by 'run-with-seed-{seed}-'.

chain_length:   int, optional (can be missing from yml)
    Suggested value 10000000
    Number of chains to use for BEAST runs.
    If not given value in template_xml_path will be used.

trace_log_every:  int > 1 or float between 0 and 1 optional (can be missing from yml)
    Suggested value 0.0001.
    How often to save a log file during BEAST runs.
    If float the save will happen as multiple of the chain_length rounded to the nearest integer.
    If not given value in template_xml_path will be used.

tree_log_every:  int > 1 or float between 0 and 1 optional (can be missing from yml)
    Suggested value 0.0001.
    How often to save a log file during BEAST runs.
    If float the save will happen as multiple of the chain_length rounded to the nearest integer.
    If not given value in template_xml_path will be used.

screen_log_every:  int > 1 or float between 0 and 1 optional (can be missing from yml)
    Suggested value 0.005.
    How often to save a log file during BEAST runs.
    If float the save will happen as multiple of the chain_length rounded to the nearest integer.
    If not given value in template_xml_path will be used.

store_state_every:  int > 1 or float between 0 and 1 optional (can be missing from yml)
    Suggested value 0.0001.
    How often to save a log file during BEAST runs.
    If float the save will happen as multiple of the chain_length rounded to the nearest integer.
    If not given value in template_xml_path will be used.


Running BEAST 2
--------------------
number_of_beast_runs: int
    Number of chains to use (repeated runs to do) when running BEAST.

beast_seeds: list of ints
    Seeds to use when running BEAST.

beast_options_without_a_value: list of strs
    Options not requiring a value to pass to BEAST 2.
     For instance to use a GPU when running BEAST 2 this would be `['-beagle_GPU']`.
    See https://www.beast2.org/2021/03/31/command-line-options.html.

beast_options_needing_a_value: dict
    Options requiring a value to pass to BEAST 2.
    For instance to use 3 threads when running BEAST 2 this would be: `{'-threads': 3}`.
    See https://www.beast2.org/2021/03/31/command-line-options.html.

sbatch_options_without_a_value: list of strs
   Options not requiring a value to pass to sbatch.
    See https://slurm.schedmd.com/sbatch.html.

sbatch_options_needing_a_value: dict
    Options requiring a value to pass to sbatch.
    See https://slurm.schedmd.com/sbatch.html.

Final Report Settings
-----------------------------
parameters_report_template: str, defaults to None
    Parameter Report to use. None value will use BDSKY-Serial.

topology: str, defaults to "CCD0"
    Topology to use for merged BEAST tree summarization and plotting, e.g. "MCC" or "CCD0".
    See BEAST2's TreeAnnotator documentation or https://www.beast2.org/2024/06/24/what-is-new-in-v2.7.7.html for more details on tree summarization methods.

summary_tree_low_memory: bool, defaults to False
    Whether to use low memory option when using BEAST 2's TreeAnnotator to summarize merged BEAST trees. Doing so will take more time. See BEAST2's TreeAnnotator documentation.

  </code>
</pre>

In [ ]:
'''
Parameters
-------------
'''
# Running an Instance of this Workflow
overall_save_dir = None
specific_run_save_dir=None
max_threads=None
kernel_name = 'beast_pype'
start_timeout = 60 # seconds to wait for a notebook to start before raising an error

# General Inputs
ready_to_go_xml = None
template_xml_path = None
fasta_path= None
metadata_path = None
sample_id_field='strain'
collection_date_field='date'
seed_to_rule_them_all = None

# Add pre-made Initial Tree
initial_tree_path = None

# Initial Tree Building & Downsampling
use_initial_tree = True
iqtree_seed = None
treetime_seed = None
downsampled_iqtree_seed = None
downsampled_treetime_seed = None
initial_tree_type = 'Temporal'
root_strain_names=None
remove_root = False
downsample_to=None

# BDSky Options
origin_start_addition = None
origin_upper_addition = None
origin_prior = None
rt_dims=None
rt_partitions=None
sampling_prop_dims=None
sampling_prop_partitions=None
zero_sampling_before_first_sample=False

# MCMC Tree/Logfile names Chain-lengths & Save Steps
log_file_basename=None
chain_length = None
trace_log_every = None
tree_log_every = None
screen_log_every = None
store_state_every = None

# Running BEAST 2
number_of_beast_runs = None
beast_seeds = None
beast_options_without_a_value=None
beast_options_needing_a_value=None
sbatch_options_without_a_value=None
sbatch_options_needing_a_value=None

# Final Report Settings
parameters_report_template = None
topology = 'CCD0'
summary_tree_low_memory = False

# Setup
## Creat Dictionary of Parameters

This needs to be done before importing packages

In [ ]:
parameters = %who_ls
parameters = {var: eval(var) for var in parameters}

## Import libraries and define functions:  

In [ ]:
import yaml
from beast_pype.nb_utils import execute_notebook
from time import perf_counter
import pandas as pd
import shutil
from beast_pype.path_utils import path_to_workflow_modules
from beast_pype.workflow_params import BDSKYSerialWorkflowParams
from beast_pype.diagnostics import gen_beast_diagnostic_nb

## Check, Setup and Record parameters

In [ ]:
parameters = BDSKYSerialWorkflowParams(**parameters)

### Creating a record for runtimes

This record list of dictionaries will be turned into a pandas dataframe and saved as a csv at the end of this notebook.

In [ ]:
runtime_records = []

### Set path to workflow modules

In [ ]:
workflow_modules_path = path_to_workflow_modules()

## Phase 2: Data Pre-Processing
### Phase 2i: Building an IQ Tree tree.

In [ ]:
#papermill_description=Phase-2i-IQTree-Building
phase_2i_params = parameters.retrieve_phase_2i_params()
if phase_2i_params is not None:
    phase_2i_start = perf_counter()
    phase_2i_IQTree_Building_log = execute_notebook(input_path=f'{workflow_modules_path}/Phase-2i-and-2iii-IQTree-Building.ipynb',
                                                    output_path=parameters.save_dir + '/Phase-2i-IQTree-Building.ipynb',
                                                    parameters=phase_2i_params,
                                                    progress_bar=True,
                                                    nest_asyncio=True, start_timeout=start_timeout
                                                    )
    phase_2i_correction_params = parameters.retrieve_phase_2i_correction_params()
    phase_2i_IQTree_Correction_log = execute_notebook(input_path=f'{workflow_modules_path}/Phase-2i-and-2iii-IQTree-Correction.ipynb',
                                                      output_path=parameters.save_dir + '/Phase-2i-IQTree-Correction.ipynb',
                                                      parameters=phase_2i_correction_params,
                                                      progress_bar=True,
                                                      nest_asyncio=True, start_timeout=start_timeout,
                                                      kernel_name=kernel_name
                                                      )

    runtime_records.append({
        'Phase': 'Phase-2i-IQTree-Building.ipynb',
        'runtime_seconds': perf_counter() - phase_2i_start
    })

### Phase 2ii: Building an TreeTime tree and Downsampling.

In [ ]:
#papermill_description=Phase-2ii-TreeTime-and-Downsampling
phase_2ii_params = parameters.retrieve_phase_2ii_params()
if phase_2ii_params is not None:
    phase_2ii_start = perf_counter()
    phase_2ii_log = execute_notebook(input_path=f'{workflow_modules_path}/Phase-2ii-and-2iv-TreeTime-and-Downsampling.ipynb',
                                     output_path=parameters.save_dir + '/Phase-2ii-TreeTime-and-Downsampling.ipynb',
                                     parameters=phase_2ii_params,
                                     progress_bar=True,
                                     nest_asyncio=True, start_timeout=start_timeout,
                                     kernel_name=kernel_name
                                     )

    runtime_records.append({
        'Phase': 'Phase-2ii-TreeTime-and-Downsampling.ipynb',
        'runtime_seconds': perf_counter() - phase_2ii_start
    })

### Phase 2iii: Building an IQ tree with Downsampled data.

In [ ]:
#papermill_description=Phase-2iii-IQTree-Building-with-Downsampled-Data
phase_2iii_params = parameters.retrieve_phase_2iii_params()
if phase_2iii_params is not None:
    phase_2iii_start = perf_counter()
    phase_2iii_IQTree_Building_log = execute_notebook(input_path=f'{workflow_modules_path}/Phase-2i-and-2iii-IQTree-Building.ipynb',
                                                      output_path=parameters.save_dir + '/Phase-2iii-IQTree-Building-with-Downsampled-Data.ipynb',
                                                      parameters=phase_2iii_params,
                                                      progress_bar=True,
                                                      nest_asyncio=True, start_timeout=start_timeout
                                                      )
    phase_2iii_correction_params = parameters.retrieve_phase_2iii_correction_params()
    phase_2iii_IQTree_Correction_log = execute_notebook(input_path=f'{workflow_modules_path}/Phase-2i-and-2iii-IQTree-Correction.ipynb',
                                                        output_path=parameters.save_dir + '/Phase-2iii-IQTree-Correction-with-Downsampled-Data.ipynb',
                                                        parameters=phase_2iii_correction_params,
                                                        progress_bar=True,
                                                        nest_asyncio=True, start_timeout=start_timeout,
                                                        kernel_name=kernel_name
                                                        )
    runtime_records.append({
        'Phase': 'Phase-2iii-IQTree-Building-with-Downsampled-Data.ipynb',
        'Sample': None,
        'Chain': None,
        'runtime_seconds': perf_counter() - phase_2iii_start
    })

### Phase 2iv: Building a TreeTime tree with Downsampled data.

In [ ]:
#papermill_description=Phase-2iv-TreeTime-with-Downsampled-Data
phase_2iv_params = parameters.retrieve_phase_2iv_params()
if phase_2iv_params is not None:
    phase_2iv_start = perf_counter()
    phase_2iv_log = execute_notebook(input_path=f'{workflow_modules_path}/Phase-2ii-and-2iv-TreeTime-and-Downsampling.ipynb',
                                     output_path=parameters.save_dir + '/Phase-2iv-TreeTime-with-Downsampled-Data.ipynb',
                                     parameters=phase_2iv_params,
                                     progress_bar=True,
                                     nest_asyncio=True, start_timeout=start_timeout,
                                     kernel_name=kernel_name
                                     )

    runtime_records.append({
        'Phase': 'Phase-2iv-TreeTime-with-Downsampled-Data.ipynb',
        'runtime_seconds': perf_counter() - phase_2iv_start
    })

## Phase 3 Generating BEAST xmls

### Placing Phase 3 Parameters in a Dictionary

In [ ]:
#papermill_description=Phase-3-Generating-XML
phase_3_params = parameters.retrieve_phase_3_params()
if phase_3_params is not None:
    phase_3_start = perf_counter()
    phase_3_log = execute_notebook(input_path=f'{workflow_modules_path}/Phase-3-Gen-BDSKY-Serial-xml.ipynb',
                                   output_path=parameters.save_dir + '/Phase-3-Gen-BDSKY-Serial-xml.ipynb',
                                   parameters=phase_3_params,
                                   progress_bar=True,
                                   nest_asyncio=True, start_timeout=start_timeout,
                                   kernel_name=kernel_name)
    runtime_records.append({
        'Phase': 'Phase-3-Gen-BDSKY-Serial-xml.ipynb',
        'runtime_seconds': perf_counter() - phase_3_start
    })

## Phase 4 Running BEAST

In [ ]:
#papermill_description=Phase-4-Running-BEAST
phase_4_start = perf_counter()
if parameters.ready_to_go_xml is not None:
    shutil.copy(parameters.ready_to_go_xml, f'{parameters.save_dir}/beast.xml')
phase_4_params = parameters.retrieve_phase_4_params()
if 'sbatch_arg_string' in phase_4_params:
    phase_4_log = execute_notebook(input_path=f'{workflow_modules_path }/Phase-4-SBATCH-Running-BEAST.ipynb',
                                   output_path=parameters.save_dir + '/Phase-4-SBATCH-Running-BEAST.ipynb',
                                   parameters=phase_4_params,
                                   progress_bar=True,
                                   nest_asyncio=True, start_timeout=start_timeout)
else:
    phase_4_log = execute_notebook(input_path=f'{workflow_modules_path }/Phase-4-GNU-Parallel-Running-BEAST.ipynb',
                                   output_path=parameters.save_dir + '/Phase-4-GNU-Parallel-Running-BEAST.ipynb',
                                   parameters=phase_4_params,
                                   progress_bar=True,
                                   nest_asyncio=True, start_timeout=start_timeout)
runtime_records.append({
        'Phase': 'Phase-4-Running-BEAST',
        'runtime_seconds': perf_counter() - phase_4_start
    })

## Phase 5: Diagnosing Outputs and Generate Report

Currently, this has to be performed manually. That being said, the code cell below will parameterize a copy of the notebook ready to run. See below for location.

In [ ]:
with open(parameters.save_dir + '/pipeline_run_info.yml', 'r') as file:
    data = file.read()
file.close()
pipeline_run_info = yaml.safe_load(data)
phase_5_params = parameters.retrieve_phase_5_params()
gen_beast_diagnostic_nb(parameters.save_dir, **phase_5_params)
print(f'Phase 5 notebook is ready for manual use at: \n{parameters.save_dir}/Phase-5-Diagnosing-Outputs-and-Generate-Report.ipynb')

## Recording Runtimes
Converting to pandas DataFrame and saving as CSV.

In [ ]:
runtime_df = pd.DataFrame.from_records(runtime_records)
runtime_df.to_csv(parameters.save_dir + "/runtimes.csv", index=False)